# COMP8851 — BWGNN on T-Finance (complete workflow)

This notebook runs the complete controlled experiment in one pass and prints progress throughout.

**Attach these two Kaggle inputs before running:**

1. `BWGNN_TFinance_COMP8851_Bundle.zip` — code/model bundle.
2. A private Kaggle dataset containing the raw DGL file `tfinance` — dataset.

Use one NVIDIA T4. If Kaggle offers T4 x2, the workflow deliberately uses GPU 0 only. Keep Internet ON for the environment-install cell.

The workflow creates fixed nested TR40/TR30/TR20/TR10 splits; runs a smoke test and an author-setting reference; tunes hidden dimension using validation AUROC only; performs 12 final runs across seeds 2, 42 and 72; records every epoch time; and creates one evidence ZIP. The raw dataset and checkpoints are not added to that ZIP.


In [ ]:
from pathlib import Path
import hashlib
import shutil
import zipfile

print("=" * 78)
print("CELL 1/4 — LOCATING BWGNN CODE AND T-FINANCE DATA")
print("=" * 78)

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working/bwgnn_tfinance")
PROJECT_DIR = WORK_ROOT / "source"
WORK_ROOT.mkdir(parents=True, exist_ok=True)

required_code = [
    "BWGNN.py",
    "main_instrumented.py",
    "generate_splits.py",
    "run_all.py",
]

if all((PROJECT_DIR / name).exists() for name in required_code):
    print(f"Existing writable project found: {PROJECT_DIR}")
else:
    bundle_zips = sorted(INPUT_ROOT.rglob("BWGNN_TFinance_COMP8851_Bundle.zip"))
    if bundle_zips:
        print(f"Code bundle found: {bundle_zips[0]}")
        with zipfile.ZipFile(bundle_zips[0]) as archive:
            archive.extractall(WORK_ROOT)
    else:
        extracted = [
            path.parent
            for path in INPUT_ROOT.rglob("run_all.py")
            if all((path.parent / name).exists() for name in required_code)
        ]
        assert extracted, (
            "BWGNN code was not found. Attach "
            "BWGNN_TFinance_COMP8851_Bundle.zip as a Kaggle input."
        )
        print(f"Automatically extracted code found: {extracted[0]}")
        if PROJECT_DIR.exists():
            shutil.rmtree(PROJECT_DIR)
        shutil.copytree(extracted[0], PROJECT_DIR)

if not all((PROJECT_DIR / name).exists() for name in required_code):
    nested_sources = [
        path.parent
        for path in WORK_ROOT.rglob("run_all.py")
        if all((path.parent / name).exists() for name in required_code)
    ]
    assert nested_sources, "The bundle was extracted, but its source folder was not found."
    if nested_sources[0] != PROJECT_DIR:
        if PROJECT_DIR.exists():
            shutil.rmtree(PROJECT_DIR)
        shutil.copytree(nested_sources[0], PROJECT_DIR)

data_names = {"tfinance", "tfinance.bin", "tfinance.dgl"}
data_candidates = [
    path
    for path in INPUT_ROOT.rglob("*")
    if path.is_file()
    and path.name.lower() in data_names
    and path.stat().st_size > 100 * 1024 * 1024
]

if not data_candidates:
    data_candidates = [
        path
        for path in Path("/kaggle/working").rglob("*")
        if path.is_file()
        and path.name.lower() in data_names
        and path.stat().st_size > 100 * 1024 * 1024
    ]

assert data_candidates, (
    "The raw T-Finance DGL file was not found. Attach a private Kaggle "
    "dataset containing a file named tfinance."
)
DATA_PATH = max(data_candidates, key=lambda path: path.stat().st_size)

def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

print(f"Writable project: {PROJECT_DIR}")
for name in required_code:
    print(f"{name}: {'FOUND' if (PROJECT_DIR / name).exists() else 'MISSING'}")
print(f"T-Finance file: {DATA_PATH}")
print(f"T-Finance size: {DATA_PATH.stat().st_size / 1024**2:.2f} MB")
print(f"T-Finance SHA256: {sha256(DATA_PATH)}")
print("PROJECT AND DATA READY: TRUE")


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

print("=" * 78)
print("CELL 2/4 — CREATING THE CUDA-ENABLED ENVIRONMENT")
print("=" * 78)

ENV_DIR = Path("/kaggle/working/bwgnn_py311")
ENV_PYTHON = ENV_DIR / "bin" / "python"

def execute(command, label):
    print(f"\n{label}")
    print("Command:", " ".join(map(str, command)))
    result = subprocess.run(command, text=True)
    assert result.returncode == 0, f"Failed: {label}"

if not ENV_PYTHON.exists():
    execute(
        [sys.executable, "-m", "pip", "install", "-q", "uv"],
        "[1/5] Installing uv",
    )
    uv_candidates = [
        shutil.which("uv"),
        "/usr/local/bin/uv",
        str(Path.home() / ".local/bin/uv"),
        str(Path(sys.executable).parent / "uv"),
    ]
    UV = next((Path(item) for item in uv_candidates if item and Path(item).exists()), None)
    assert UV is not None, "uv was installed but its executable could not be located."
    execute(
        [str(UV), "venv", "--python", "3.11", "--seed", str(ENV_DIR)],
        "[2/5] Creating the Python 3.11 virtual environment",
    )
else:
    print(f"[1–2/5] Existing Python environment found: {ENV_DIR}")
    uv_candidates = [
        shutil.which("uv"),
        "/usr/local/bin/uv",
        str(Path.home() / ".local/bin/uv"),
        str(Path(sys.executable).parent / "uv"),
    ]
    UV = next((Path(item) for item in uv_candidates if item and Path(item).exists()), None)
    if UV is None:
        execute([sys.executable, "-m", "pip", "install", "-q", "uv"], "Restoring uv")
        UV = next(
            Path(item)
            for item in [shutil.which("uv"), "/usr/local/bin/uv", str(Path.home() / ".local/bin/uv")]
            if item and Path(item).exists()
        )

execute(
    [
        str(UV), "pip", "install", "--python", str(ENV_PYTHON),
        "torch==2.2.2", "--index-url", "https://download.pytorch.org/whl/cu121",
    ],
    "[3/5] Installing CUDA-enabled PyTorch",
)

execute(
    [
        str(UV), "pip", "install", "--python", str(ENV_PYTHON),
        "dgl==1.1.3+cu121", "--find-links", "https://data.dgl.ai/wheels/cu121/repo.html",
    ],
    "[4/5] Installing CUDA-enabled DGL",
)

execute(
    [
        str(UV), "pip", "install", "--python", str(ENV_PYTHON),
        "numpy==1.26.4", "scipy==1.12.0", "scikit-learn==1.4.2",
        "sympy==1.12", "packaging==24.2", "setuptools==75.8.0", "psutil==7.0.0",
    ],
    "[5/5] Installing the remaining pinned dependencies",
)

verification = r"""
import dgl
import torch

print('Python/PyTorch/DGL:', torch.__version__, dgl.__version__)
print('CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'CUDA is unavailable in the experiment environment.'
print('GPU:', torch.cuda.get_device_name(0))
print('GPU memory GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
graph = dgl.graph((torch.tensor([0, 1]), torch.tensor([1, 0])))
graph = graph.to('cuda:0')
assert graph.device.type == 'cuda'
print('CUDA DGL graph test: PASS')
"""

environment = os.environ.copy()
environment["CUDA_VISIBLE_DEVICES"] = "0"
environment["DGLBACKEND"] = "pytorch"
result = subprocess.run([str(ENV_PYTHON), "-c", verification], env=environment, text=True)
assert result.returncode == 0, "The PyTorch/DGL GPU verification failed."
print("ENVIRONMENT READY: TRUE")


In [ ]:
from pathlib import Path
import os
import subprocess
import time

print("=" * 78)
print("CELL 3/4 — RUNNING THE COMPLETE BWGNN × T-FINANCE WORKFLOW")
print("=" * 78)

PROJECT_DIR = Path("/kaggle/working/bwgnn_tfinance/source")
ENV_PYTHON = Path("/kaggle/working/bwgnn_py311/bin/python")
OUTPUT_ZIP = Path("/kaggle/working/bwgnn_tfinance_complete_results.zip")

assert (PROJECT_DIR / "run_all.py").exists(), "Run Cell 1 again."
assert ENV_PYTHON.exists(), "Run Cell 2 again."

if "DATA_PATH" not in globals() or not Path(DATA_PATH).exists():
    names = {"tfinance", "tfinance.bin", "tfinance.dgl"}
    found = [
        path
        for path in Path("/kaggle/input").rglob("*")
        if path.is_file() and path.name.lower() in names and path.stat().st_size > 100 * 1024 * 1024
    ]
    assert found, "The T-Finance input is no longer attached; rerun Cell 1."
    DATA_PATH = max(found, key=lambda path: path.stat().st_size)

command = [
    str(ENV_PYTHON), "-u", str(PROJECT_DIR / "run_all.py"),
    "--data-path", str(DATA_PATH),
    "--output-zip", str(OUTPUT_ZIP),
]

environment = os.environ.copy()
environment["CUDA_VISIBLE_DEVICES"] = "0"
environment["DGLBACKEND"] = "pytorch"
environment["PYTHONUNBUFFERED"] = "1"

print("Model: BWGNN")
print("Dataset: T-Finance")
print("GPU: device 0 only")
print("Final ratios: TR40, TR30, TR20, TR10")
print("Final seeds: 2, 42, 72")
print("Progress and per-epoch information will appear below.")
print("Command:", " ".join(command))
print("-" * 78)

started = time.time()
process = subprocess.Popen(
    command,
    cwd=PROJECT_DIR,
    env=environment,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end="", flush=True)

return_code = process.wait()
elapsed_minutes = (time.time() - started) / 60
print("-" * 78)
print(f"Workflow return code: {return_code}")
print(f"Elapsed time this execution: {elapsed_minutes:.2f} minutes")
assert return_code == 0, (
    "The workflow stopped. Read the final displayed error. If the Kaggle "
    "session is still active, fix that error and rerun this cell; completed runs will be skipped."
)
assert OUTPUT_ZIP.exists(), "The workflow ended without creating the evidence ZIP."
print("COMPLETE WORKFLOW CELL: TRUE")


In [ ]:
from pathlib import Path
import csv
import hashlib
import json
import zipfile
from IPython.display import FileLink, display

print("=" * 78)
print("CELL 4/4 — VERIFYING AND DOWNLOADING THE FINAL RESULTS")
print("=" * 78)

OUTPUT_ZIP = Path("/kaggle/working/bwgnn_tfinance_complete_results.zip")
PROJECT_DIR = Path("/kaggle/working/bwgnn_tfinance/source")
REPORT_DIR = PROJECT_DIR / "reports"

assert OUTPUT_ZIP.exists(), "The final ZIP is missing. Run Cell 3."
with zipfile.ZipFile(OUTPUT_ZIP) as archive:
    bad = archive.testzip()
    names = archive.namelist()
assert bad is None, f"ZIP integrity failure: {bad}"

digest = hashlib.sha256(OUTPUT_ZIP.read_bytes()).hexdigest()
status = json.loads((REPORT_DIR / "verification_status.json").read_text())
selection = json.loads((REPORT_DIR / "tuning_selection.json").read_text())

print(f"Model: BWGNN")
print(f"Dataset: T-Finance")
print(f"Selected hidden dimension: {selection['selected_hidden_dimension']}")
print(f"Tuning used test metrics: {selection['test_metrics_used_for_selection']}")
print(f"Final runs found: {status['final_runs_found']}/{status['final_runs_expected']}")
print(f"Recorded final epoch rows: {status['final_epoch_rows']}")
print(f"Files in final ZIP: {len(names)}")
print(f"ZIP size: {OUTPUT_ZIP.stat().st_size / 1024**2:.2f} MB")
print(f"ZIP SHA256: {digest}")

print("\nTHREE-SEED MEAN ± SAMPLE SD")
with (REPORT_DIR / "final_results_mean_sd.csv").open() as handle:
    rows = list(csv.DictReader(handle))
for row in rows:
    print(
        f"{row['ratio']}: "
        f"AUROC {float(row['auroc_mean']):.4f} ± {float(row['auroc_sample_sd']):.4f} | "
        f"AUPRC {float(row['auprc_mean']):.4f} ± {float(row['auprc_sample_sd']):.4f} | "
        f"Macro-F1 {float(row['macro_f1_mean']):.4f} ± {float(row['macro_f1_sample_sd']):.4f} | "
        f"Mean epoch {float(row['mean_epoch_seconds_mean']):.3f}s"
    )

print("\nFINAL EVIDENCE PACKAGE READY: TRUE")
print("Download this file and send it back for verification and document/GitHub packaging:")
display(FileLink(str(OUTPUT_ZIP)))
